Este es el notebook para el ejercicio propuesto en el TP Final de la materia Vision por Computadora.

In [1]:
pip install opencv-python

1. Montar el drive de Google para leer los datos de la carpeta correspondiente.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


La siguiente celda permite elegir una carpeta para cargar las imágenes de las emociones. Para gestionar efectivamente la memoria se aplica el preprocesamiento convirtiendo a escala de grises y redimensionando de acuerdo a parámetros establecidos por el usuario


In [2]:
import os
import cv2
import matplotlib.pyplot as plt

# Pedir al usuario la ruta de la carpeta principal que contiene las carpetas de emociones
image_base_folder_path = input("Por favor, introduce la ruta completa a la carpeta principal que contiene las carpetas de emociones (ej: /content/drive/MyDrive/00.DataVxC/emotions_dataset/): ")

# Solicitar al usuario las dimensiones deseadas para las imágenes pre-procesadas
while True:
    try:
        target_width = int(input("Introduce el ancho deseado para las imágenes pre-procesadas (ej: 48): "))
        target_height = int(input("Introduce el alto deseado para las imágenes pre-procesadas (ej: 48): "))
        if target_width > 0 and target_height > 0:
            break
        else:
            print("Las dimensiones deben ser números positivos. Intenta de nuevo.")
    except ValueError:
        print("Entrada inválida. Por favor, introduce un número entero.")

target_dim = (target_width, target_height)
print(f"Las imágenes se pre-procesarán a dimensiones: {target_dim} en escala de grises.")

# Verificar si la carpeta base existe
if not os.path.isdir(image_base_folder_path):
    print(f"Error: La carpeta '{image_base_folder_path}' no existe. Por favor, verifica la ruta.")
else:
    print(f"Procesando imágenes de las subcarpetas en: {image_base_folder_path}")
    processed_images = [] # Lista para almacenar las imágenes ya procesadas
    image_filenames = [] # Lista para almacenar los nombres de archivo relativos
    image_labels = [] # Lista para almacenar las etiquetas de las emociones

    # Extensiones de imagen comunes
    image_extensions = ('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff')

    # Iterar sobre los elementos dentro de la carpeta base
    for emotion_folder_name in os.listdir(image_base_folder_path):
        emotion_folder_path = os.path.join(image_base_folder_path, emotion_folder_name)

        # Verificar si es un directorio
        if os.path.isdir(emotion_folder_path):
            emotion_label = emotion_folder_name # El nombre de la carpeta es la etiqueta de la emoción
            print(f"  Cargando y pre-procesando imágenes para la emoción: {emotion_label}")

            for filename in os.listdir(emotion_folder_path):
                if filename.lower().endswith(image_extensions):
                    img_path = os.path.join(emotion_folder_path, filename)
                    try:
                        img = cv2.imread(img_path) # Cargar la imagen
                        if img is not None:
                            # Convertir a escala de grises
                            gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

                            # Redimensionar la imagen
                            resized_img = cv2.resize(gray_img, target_dim, interpolation=cv2.INTER_AREA)

                            processed_images.append(resized_img)
                            image_filenames.append(os.path.join(emotion_folder_name, filename)) # Guardar la ruta relativa
                            image_labels.append(emotion_label)
                            # Opcional: mostrar un progreso
                            # print(f"    Cargada y procesada: {os.path.join(emotion_folder_name, filename)}")
                        else:
                            print(f"    Advertencia: No se pudo cargar la imagen '{filename}' en '{emotion_folder_name}'. Posiblemente corrupta o formato no soportado.")
                    except Exception as e:
                        print(f"    Error al cargar o procesar la imagen '{filename}' en '{emotion_folder_name}': {e}")

    print(f"Total de imágenes cargadas y pre-procesadas: {len(processed_images)} de {image_base_folder_path}")

    # Las imágenes procesadas (en escala de grises y redimensionadas) están ahora en la lista 'processed_images',
    # sus nombres de archivo (relativos a la carpeta de emoción) en 'image_filenames',
    # y sus etiquetas de emoción en 'image_labels'.
    # Puedes acceder a ellas así:
    # for i, img_proc in enumerate(processed_images):
    #     print(f"Procesando imagen: {image_filenames[i]} (Emoción: {image_labels[i]}) con dimensiones {img_proc.shape}")

    # Ejemplo de visualización de las primeras 5 imágenes procesadas (opcional)
    # if processed_images:
    #     plt.figure(figsize=(15, 5))
    #     for i in range(min(5, len(processed_images))):
    #         plt.subplot(1, 5, i + 1)
    #         # Mostrar en escala de grises
    #         plt.imshow(processed_images[i], cmap='gray')
    #         plt.title(f"{image_filenames[i]} ({image_labels[i]})")
    #         plt.axis('off')
    #     plt.show()


Por favor, introduce la ruta completa a la carpeta principal que contiene las carpetas de emociones (ej: /content/drive/MyDrive/00.DataVxC/emotions_dataset/): /content/drive/MyDrive/00.DataVxC
Introduce el ancho deseado para las imágenes pre-procesadas (ej: 48): 48
Introduce el alto deseado para las imágenes pre-procesadas (ej: 48): 48
Las imágenes se pre-procesarán a dimensiones: (48, 48) en escala de grises.
Procesando imágenes de las subcarpetas en: /content/drive/MyDrive/00.DataVxC
  Cargando y pre-procesando imágenes para la emoción: Angry
  Cargando y pre-procesando imágenes para la emoción: Fear
  Cargando y pre-procesando imágenes para la emoción: Happy
  Cargando y pre-procesando imágenes para la emoción: Sad
  Cargando y pre-procesando imágenes para la emoción: Suprise
Total de imágenes cargadas y pre-procesadas: 59099 de /content/drive/MyDrive/00.DataVxC


Aplica el clasificador (la opcion 3 es la que mas rostros detecta)

In [6]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

# Obtener la ruta base de los clasificadores Haar Cascade
cascades_path = cv2.data.haarcascades

# Listar los clasificadores de rostros frontales disponibles
face_classifiers_options = [
    'haarcascade_frontalface_default.xml',
    'haarcascade_frontalface_alt.xml',
    'haarcascade_frontalface_alt2.xml',
    'haarcascade_frontalface_alt_tree.xml'
]

print("Clasificadores de rostros disponibles:")
for i, classifier_name in enumerate(face_classifiers_options):
    print(f"{i+1}. {classifier_name}")

selected_classifier_index = -1
while not (0 < selected_classifier_index <= len(face_classifiers_options)):
    try:
        selected_classifier_index = int(input(f"Selecciona el número del clasificador de rostros que deseas usar (1-{len(face_classifiers_options)}): "))
        if not (0 < selected_classifier_index <= len(face_classifiers_options)):
            print("Selección inválida. Por favor, introduce un número dentro del rango.")
    except ValueError:
        print("Entrada inválida. Por favor, introduce un número entero.")

selected_classifier_name = face_classifiers_options[selected_classifier_index - 1]
face_cascade_path = os.path.join(cascades_path, selected_classifier_name)

# Verificar si el archivo cascade existe
if not os.path.exists(face_cascade_path):
    print(f"Error: El clasificador '{selected_classifier_name}' no se encontró en '{face_cascade_path}'.")
    print("Por favor, asegúrate de que OpenCV esté correctamente instalado y el archivo cascade esté disponible.")
    # Puedes intentar descargar el archivo si es necesario, por ejemplo:
    # !wget -q https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/{selected_classifier_name} -P {cascades_path}
    # if not os.path.exists(face_cascade_path):
    #     print("Descarga fallida o ruta incorrecta. No se puede proceder con la detección de rostros.")
    #     face_cascade = None # Para evitar errores posteriores
    # else:
    #     print(f"Haar Cascade '{selected_classifier_name}' descargado con éxito.")
else:
    print(f"Usando el clasificador: {selected_classifier_name}")

# Cargar el clasificador de rostros
face_cascade = cv2.CascadeClassifier(face_cascade_path) if os.path.exists(face_cascade_path) else None

detected_faces = []
detected_faces_labels = []

if face_cascade is None:
    print("No se pudo cargar el clasificador de rostros. No se realizará la detección de rostros.")
elif 'processed_images' in locals() and len(processed_images) > 0:
    print(f"Iniciando detección de rostros en {len(processed_images)} imágenes...")
    for i, img_proc in enumerate(processed_images):
        # img_proc ya está en escala de grises y redimensionada

        # Realizar la detección de rostros
        # scaleFactor: Parámetro que especifica cuánto se reduce la imagen en cada escala de imagen.
        # minNeighbors: Parámetro que especifica cuántos vecinos debe tener cada rectángulo candidato para retenerlo.
        # minSize: Tamaño mínimo posible del objeto. Los objetos más pequeños se ignoran.
        faces = face_cascade.detectMultiScale(img_proc, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

        if len(faces) > 0:
            for (x, y, w, h) in faces:
                # Recortar el rostro detectado
                face_roi = img_proc[y:y+h, x:x+w]

                # Opcional: Redimensionar los rostros detectados a un tamaño uniforme si se desea
                # Por ejemplo, si los rostros se van a alimentar a un modelo que espera un tamaño fijo,
                # aunque las imágenes ya están redimensionadas, el recorte podría variar.
                # face_roi = cv2.resize(face_roi, target_dim, interpolation=cv2.INTER_AREA)

                detected_faces.append(face_roi)
                # Asignar la etiqueta de emoción original a cada rostro detectado
                detected_faces_labels.append(image_labels[i])

            # Opcional: dibujar rectángulos alrededor de los rostros para depuración
            # for (x, y, w, h) in faces:
            #     cv2.rectangle(img_proc, (x, y), (x+w, y+h), (255, 0, 0), 2) # Rojo en imagen gris (ej: (255) para blanco, (0) para negro)
            # print(f"  Rostros detectados en imagen {i+1} ({image_filenames[i]}). Total: {len(faces)}")
        # else:
        #     print(f"  No se detectaron rostros en imagen {i+1} ({image_filenames[i]})")

    print(f"Detección de rostros completada. Total de rostros detectados: {len(detected_faces)}")

    # Los rostros detectados están ahora en la lista 'detected_faces',
    # y sus etiquetas de emoción correspondientes están en 'detected_faces_labels'.
    # Puedes acceder a ellos así:
    # for i, face in enumerate(detected_faces):
    #     print(f"Rostro detectado {i+1} (Emoción: {detected_faces_labels[i]}) con dimensiones {face.shape}")

    # Ejemplo de visualización de los primeros 5 rostros detectados (opcional)
    # if detected_faces:
    #     plt.figure(figsize=(15, 5))
    #     for i in range(min(5, len(detected_faces))):
    #         plt.subplot(1, 5, i + 1)
    #         plt.imshow(detected_faces[i], cmap='gray')
    #         plt.title(f"{detected_faces_labels[i]}")
    #         plt.axis('off')
    #     plt.show()

else:
    print("No se encontraron imágenes procesadas para la detección de rostros. Por favor, asegúrate de haber ejecutado la celda anterior (`rtaw70B6K8Kb`) para cargar y pre-procesar las imágenes primero.")


Clasificadores de rostros disponibles:
1. haarcascade_frontalface_default.xml
2. haarcascade_frontalface_alt.xml
3. haarcascade_frontalface_alt2.xml
4. haarcascade_frontalface_alt_tree.xml
Selecciona el número del clasificador de rostros que deseas usar (1-4): 4
Usando el clasificador: haarcascade_frontalface_alt_tree.xml
Iniciando detección de rostros en 59099 imágenes...
Detección de rostros completada. Total de rostros detectados: 2076
